# NB30 — Scroll trajectory beyond viewport analytics

**Regime:** `[LAB]`. Target label (NB22 gaze-regression) requires gaze.
The trajectory + viewport features themselves are cursor-free and derived
from scroll events + DOM bboxes — `[BOTH]`-eligible in principle.

## Question

**Viewport analytics** — continuous visibility, time-in-view, time near
viewport center — is the conventional cursor-free attention probe for
feed ranking (MRC viewability definitions; feed-ranking viewport signals
at platform-scale operators).
On SERPs, does **scroll trajectory** (velocity kinematics, pauses,
direction reversals, deceleration near AOI center) add discriminative
power for the deferred-vs-evaluated-rejected split *beyond* what
viewport analytics already provides?

## Motivation

- NB17 reported a scroll null: scroll dwell / velocity / pause duration
  do not differ between clicked vs not-clicked on desktop (all *p* > 0.3).
  That test used raw features against the click outcome without any
  incremental-AUC testing against a viewport baseline.
- NB28 established that a banded viewport decomposition (top/mid/bot
  thirds) reaches AUC 0.799 on deferred-vs-rejected. The bands are a
  research operationalization — continuous viewport analytics is the
  conventional instrument, and it must be the baseline a scroll-
  trajectory claim has to beat, not the raw click outcome NB17 used.

This notebook runs the incremental-AUC test. Trajectory adding measurable AUC on top of continuous viewport analytics
would provide a second cursor-free evaluation probe that works on mobile
(no cursor) and on feeds (no SERP structure). No detectable signal would
tighten NB17's desktop scroll null.

## Feature sets

- **A — Viewport bands** (NB28 operationalization, 4 features):
  `vt_any, vt_top, vt_mid, vt_bot` (per-AOI ms in each viewport third).
- **B — Continuous viewport analytics** (the baseline, 4 features):
  `vt_any, vt_center_ms` (time with AOI center within ±100 px of viewport
  center), `avg_viewport_y` (mean AOI-center viewport-y during visibility),
  `max_overlap_frac` (max fraction of AOI visible).
- **C — Scroll trajectory** (the test, 7 features):
  `max_abs_velocity, min_abs_velocity, pause_ms, n_reversals,
  max_decel_near_center, entry_velocity, exit_velocity`.

## Protocol

Population: approached (`min_dist` < 100 px) ∧ ¬clicked, same as NB28.
Label: NB22 gaze-regression (1 = deferred, 0 = eval-rejected).
Classifier: sklearn `LogisticRegression` with `class_weight='balanced'`.
CV: 47-fold leave-one-participant-out. Headline = paired per-participant
Wilcoxon on held-out AUCs.

Headline contrast: `B ∪ C > B`. Does trajectory add on top of industry-
standard continuous viewport analytics?


## Key Claims (authoritative for paper writers)

*Last verified against executed notebook output: 2026-04-12.*
*Notebook: `30_scroll_trajectory.ipynb`.*

If prose in a paper draft cites a value that disagrees with a row below, the paper is wrong — not the notebook. If re-running this notebook produces different values, update this block immediately and `grep` for the old value across `docs/`.

### Dataset and population

**Regime:** `[LAB]`. Target label (NB22 gaze-regression) is LAB-only; the features (scroll trajectory + viewport analytics) are cursor-free and `[BOTH]`-eligible in principle. 47-fold leave-one-participant-out LR.

| ID | Claim | Value |
|---|---|---|
| **K1** | Rows (approached ∧ ¬clicked) used for the deferred-vs-rejected test | **2,351** |
| **K2** | Deferred (NB22 gaze-regression = 1) | 1,916 |
| **K3** | Eval-rejected (NB22 gaze-regression = 0) | 435 |

### Feature sets

- **A** — Viewport bands (NB28): `vt_any, vt_top, vt_mid, vt_bot` (4 features)
- **B** — Continuous viewport analytics (the baseline): `vt_any, vt_center_ms, avg_viewport_y, max_overlap_frac` (4 features)
- **C** — Scroll trajectory: `max_abs_velocity, min_abs_velocity, pause_ms, n_reversals, max_decel_near_center, entry_velocity, exit_velocity` (7 features)

### Pooled LOPO AUC (deferred-vs-rejected)

| ID | Scorer | Pooled AUC | Per-participant mean ± SD |
|---|---|---:|---:|
| **K4** | A bands (NB28) | **0.7990** | 0.8051 ± 0.1272 |
| **K5** | B continuous viewport | **0.7978** | 0.8107 ± 0.1204 |
| **K6** | C trajectory alone | 0.7496 | 0.7559 ± 0.1252 |
| **K7** | A ∪ C | 0.8104 | 0.8179 ± 0.1138 |
| **K8** | B ∪ C | **0.8168** | 0.8293 ± 0.1122 |
| **K9** | A ∪ B ∪ C | 0.8200 | 0.8326 ± 0.1090 |

### Paired per-participant Wilcoxon signed-rank (one-sided, 47 participants)

| ID | Comparison | Δ ± SD | Consistency | p |
|---|---|---:|---:|---:|
| **K10** | C > A (trajectory alone beats bands?) | −0.0492 ± 0.1024 | 14 / 47 | 0.9993 (ns) |
| **K11** | C > B (trajectory alone beats continuous viewport?) | −0.0548 ± 0.1127 | 12 / 47 | 0.9997 (ns) |
| **K12** | A ∪ C > A (trajectory added to bands) | **+0.0128 ± 0.0417** | 29 / 47 | **0.0368** |
| **K13** | **B ∪ C > B (trajectory added to continuous viewport) — headline** | **+0.0185 ± 0.0456** | **36 / 47** | **0.0031** |
| **K14** | A ∪ B ∪ C > B ∪ C (bands add on top of B + C?) | +0.0033 ± 0.0232 | 29 / 47 | 0.2178 (ns) |
| **K15** | B ∪ C > B with trajectory re-computed on **click-time-truncated** scroll timeline (leakage check) | **+0.0200 ± 0.0493** | 34 / 47 | **0.0038** |

### Pre-implementation ablations (2026-04-19)

Run before freezing the feature set the approach-retreat JS library will emit. Purpose: find the parsimonious B∪C' subset that recovers the K13 lift, verify the feature set is linearly complete, and test how much the lift degrades if browser `scroll` events fire at low rates.

| ID | Claim | Value |
|---|---|---|
| **K16** | LOFO on C under Holm–Bonferroni (α = 0.05, 7 tests): features whose removal significantly hurts per-p AUC | **1 of 7**: `min_abs_velocity` (Δ = +0.0174 on drop, Holm p = 0.012). The other 6 are **drop-candidates** under LOFO (Holm p ≥ 0.12 each) — LOFO is conservative when features carry substitutable signal. |
| **K17** | Redundancy: Spearman pairs with \|r\| ≥ 0.85 on the 2,351-row sample | **1 pair**: `pause_ms` ↔ `vt_any` **r = +0.995**. VIF confirms: `vt_any` VIF = 237, `pause_ms` VIF = 226. Must drop one of the two before emitting. |
| **K18** | Greedy forward-selection from B: features added until the next-best candidate fails paired one-sided Wilcoxon at p < 0.10 | **`min_abs_velocity`** (step 1, Δ = +0.0077, p = 0.039) then **`n_reversals`** (step 2, Δ = +0.0097, p = 0.038). Step 3's best candidate `entry_velocity` does not clear (p = 0.28). |
| **K19** | Minimal B ∪ C' (6 features) vs full B ∪ C (11 features) | pooled AUC 0.8143 vs 0.8168 (Δ_pooled = −0.0025); paired per-p Δ = +0.0011, **p = 0.356 (ns)**. **The other 5 trajectory features add no detectable AUC once `min_abs_velocity` and `n_reversals` are present.** |
| **K20** | Event-rate sensitivity: paired Δ(B ∪ C − B) per-p as scroll events are decimated | native +0.0185 (p = 0.003); 30 Hz +0.0113 (p = 0.037); 10 Hz +0.0045 (p = 0.059 ns); 5 Hz +0.0079 (p = 0.013); 2 Hz +0.0101 (p = 0.020). Signal **persists at 2 Hz** — degrades but does not disappear. The 10 Hz non-significance is within paired-Wilcoxon noise (adjacent rates are significant). |
| **K21** | LGBM vs LR on full B ∪ C | LGBM pooled AUC 0.8107 vs LR 0.8168; paired per-p Δ(LGBM − LR) = **−0.0135, p = 0.90 (ns, LR wins)**. The 11-feature set is **LR-complete** — trees do not surface useful interactions to add as emitted features. |

### Recommended emission for the approach-retreat library

Based on K16–K21, the minimal, non-redundant, LR-complete emission beyond the existing M4 cursor features is **4 B (continuous viewport) + 2 C (trajectory) = 6 features**:

1. `vt_any` — time AOI overlapped the viewport (ms)
2. `vt_center_ms` — time AOI center was within ±100 px of viewport center (ms)
3. `avg_viewport_y` — mean viewport-y of AOI center during visibility (px)
4. `max_overlap_frac` — peak fraction of AOI visible within viewport
5. `min_abs_velocity` — minimum |scroll velocity| during AOI visibility (px/s)
6. `n_reversals` — scroll direction reversals during AOI visibility (count)

The banded decomposition (A: `vt_top`, `vt_mid`, `vt_bot`) can stay behind an opt-in flag for backward compatibility but is not default. `pause_ms` is collinear with `vt_any` and is not emitted.


### What the extension shows

- **K13 is the headline.** Trajectory features add a paired Δ of +0.0185 AUC (p = 0.003, 36/47 participants) on top of the continuous-viewport analytics baseline (B). Scroll kinematics are not redundant with where-the-AOI-sat-in-the-viewport: they carry an additional cursor-free signal for the deferred-vs-rejected split.
- **Trajectory alone is insufficient.** K10 and K11 show that trajectory features without a visibility baseline perform worse than either viewport scorer. The useful signal is *incremental given a visibility baseline*; it is not self-standing.
- **At n=47 the banded decomposition provides no detectable additional AUC beyond B∪C (K14).** A∪B∪C vs B∪C paired Δ = +0.003 (p = 0.22, ns) — directionally positive but below detection threshold at this sample size. We cannot rule in or out a small effect with n=47; we *can* say the banded split is not load-bearing for the classifier at the sample sizes the paper is reported at. **Classifier parsimony is not dashboard parsimony, however.** The banded decomposition remains the natural visualization surface for a commercial viewport-analytics tool — per-result band-time heatmaps, top/mid/bot residency strips, and deferred-vs-rejected prototype comparisons are human-readable where a continuous `avg_viewport_y_px` scalar is not. The `approach-retreat` library emits both: bands default-on behind `trackViewportBands`, continuous analytics via `getViewportAnalytics()`. Ranker consumes the six-feature set; analyst dashboard consumes bands.
- **Leakage check (K15) — no detectable click-settle contamination.** Re-computing the trajectory features on a scroll timeline truncated at the click timestamp (so post-click settle-scroll cannot contribute to features like `exit_velocity` for non-clicked AOIs in the trial) reproduces the headline paired Δ within 0.0014 AUC (+0.0200 truncated vs +0.0185 full, both p < 0.005). The K13 lift is not an artifact of using the full-trial scroll timeline.
- **NB17 vs K13 — complementary, not a refutation.** NB17 tested univariate scroll dwell/velocity/pause between clicked vs not-clicked and found non-significant differences (all p > 0.3). NB30 tests a different target (deferred vs eval-rejected on approached∧¬clicked records) with a joint-model incremental-AUC test. Three axes differ (target, feature set, test statistic); NB30 is a new finding at a different site, not an overturning of NB17's null.

### CENTER_TOL sensitivity (2026-04-19)

| ID | Claim | Value |
|---|---|---|
| **K22** | CENTER_TOL sensitivity sweep on the minimal 6-feature set: pooled AUC at {25, 50, 100, 200, 400} px | {0.8142, 0.8140, 0.8143, 0.8140, 0.8151} — **flat across 16× range** (spread 0.001). Canonical 100 px is fine to freeze. |

### Windowing — cumulative vs rolling (2026-04-19)

| ID | Claim | Value |
|---|---|---|
| **K23** | Windowing — rolling 5-second window ending at click_t vs cumulative-since-AOI-first-seen | Cumulative B pooled 0.798 → rolling 0.704; cumulative B∪C pooled 0.817 → rolling 0.698. Paired Δ(cumulative − rolling) = **+0.1194 per-p** (43/47, **p < 0.0001**). Within the rolling window, B∪C > B is null (Δ = −0.005, p = 0.86). **The deferred-vs-rejected signal requires cumulative accumulation**; a 5-second window drops the features to near-chance. |

### EWM empirical signatures (2026-04-19) — feeds §4.5 of the CIKM draft

Direct Mann–Whitney two-sided tests of the raw features on the approached ∧ ¬clicked subset. Predictions from Gray & Fu 2004 / SCH framing (viewport as the external working memory buffer, trajectory features as EWM management actions).

| ID | Claim | Value |
|---|---|---|
| **K24** | `n_reversals` deferred vs eval-rejected (EWM-reload-consistent) | Deferred mean = 1.97, eval-rejected = 1.12; **Δ = +0.86, p = 8.5 × 10⁻²³**. Deferred items accumulate ~1 additional scroll direction reversal during AOI visibility. |
| **K25** | `min_abs_velocity` deferred vs eval-rejected (viewport stabilization) | Deferred mean = 0.07 px/s, eval-rejected = 0.31 px/s; deferred median = 0 (user at a full stop), eval-rejected median = 0.10 px/s. **Δ = −0.24 px/s, p = 2.0 × 10⁻⁴⁷**. |
| **K26** | `avg_viewport_y` deferred vs eval-rejected (position in viewport) | Deferred mean = 453 px, eval-rejected = 622 px. **Δ = −169 px, p = 1.5 × 10⁻⁴²**. Deferred items spend more time near viewport *top* (where they reappear after a scroll-back); eval-rejected items near viewport bottom. |

### Deployability

- Features transfer to mobile and to feed-style layouts in principle — scroll events + DOM bboxes are available in both. Validation on ACD (no per-AOI SERP structure) or a mobile feed dataset is future work.
- For deployment, the **minimal 6-feature set** (NB30:K18 forward-selection: 4 continuous viewport + `min_abs_velocity` + `n_reversals`) recovers the full 11-feature B∪C lift within +0.001 AUC (K19). The `approach-retreat` JS library emits these 6 features per AOI at session granularity; see its `getViewportAnalytics()` API.

### Peter Dixon-Moses cursor-as-fourth-viewport extension (2026-04-19)

| ID | Claim | Value |
|---|---|---|
| **K27** | Joint forward selection from B with candidates `C ∪ M4` (including `dwell_in_proximity_ms`, the cursor-viewport-residence feature Peter proposed — time with `|cursor_page_y − aoi_center_y| < 100` px) | Step 1 picks **`mean_dist`** (M4, Δ = +0.0163, *p* = 0.012); step 2 picks **`min_abs_velocity`** (C, Δ = +0.0078, *p* = 0.043); step 3 stops (best *p* = 0.159). **`dwell_in_proximity_ms` was not picked.** The binary in-band/out-of-band formulation loses to `mean_dist` (continuous mean cursor proximity), which captures the same signal with more gradient. Final B + `mean_dist` + `min_abs_velocity` = 6 features, per-p 0.8348. Vs NB30:K18 canonical (B + `min_abs_velocity` + `n_reversals`, per-p 0.8282): paired Δ = +0.0066, *p* = 0.20 ns. Peter's cursor-viewport intuition is correct in substance — cursor proximity is load-bearing — but the empirically strongest encoding is continuous mean distance, not time-in-band. |


In [1]:
# ── Imports, paths, and feature computation ─────────────────────────
import json
import sys
import warnings
from pathlib import Path

import numpy as np
from scipy.stats import wilcoxon
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=UserWarning)

ROOT = Path('/Users/andyed/Documents/dev/attentional-foraging')
sys.path.insert(0, str(ROOT / 'notebooks-v2'))
sys.path.insert(0, str(ROOT / 'scripts'))
from data_loader import get_trial_meta, load_mouse_events, result_bands  # noqa
from nb30_scroll_trajectory import compute_features_for_trial, FEATURES_A, FEATURES_B, FEATURES_C

FEATURES_JSON = ROOT / 'AdSERP/data/cursor-approach-features.json'
REG_CACHE = ROOT / 'scripts/output/approach_threshold_sensitivity/regression_labels_cache.json'
print('imports ok')
print(f'A bands ({len(FEATURES_A)}):         {FEATURES_A}')
print(f'B continuous viewport ({len(FEATURES_B)}): {FEATURES_B}')
print(f'C trajectory ({len(FEATURES_C)}):          {FEATURES_C}')

imports ok
A bands (4):         ['vt_any', 'vt_top', 'vt_mid', 'vt_bot']
B continuous viewport (4): ['vt_any', 'vt_center_ms', 'avg_viewport_y', 'max_overlap_frac']
C trajectory (7):          ['max_abs_velocity', 'min_abs_velocity', 'pause_ms', 'n_reversals', 'max_decel_near_center', 'entry_velocity', 'exit_velocity']


In [2]:
# ── Compute features for every trial × position ─────────────────────
raw = json.load(open(FEATURES_JSON))
labels = np.array(json.load(open(REG_CACHE)), dtype=bool)
assert len(labels) == len(raw)

trials = sorted({r['trial_id'] for r in raw})
print(f'trials to process: {len(trials):,}')

per_trial = {}
missing = 0
for i, tid in enumerate(trials):
    feats = compute_features_for_trial(tid, n_positions=10)
    if feats is None:
        missing += 1
        continue
    per_trial[tid] = feats
    if (i + 1) % 400 == 0:
        print(f'  {i+1}/{len(trials)} (missing so far: {missing})')
print(f'done. computed: {len(per_trial):,}  missing meta/events: {missing}')

trials to process: 2,339


  400/2339 (missing so far: 0)


  800/2339 (missing so far: 0)


  1200/2339 (missing so far: 0)


  1600/2339 (missing so far: 0)


  2000/2339 (missing so far: 0)


done. computed: 2,339  missing meta/events: 0


In [3]:
# ── Join to label rows + subset to approached ∧ ¬clicked ─────────────
keep_idx, feat_rows = [], []
for i, r in enumerate(raw):
    tid = r['trial_id']
    pos = r['position']
    if tid not in per_trial or pos >= 10:
        continue
    feat_rows.append(per_trial[tid][pos])
    keep_idx.append(i)
keep_idx = np.array(keep_idx)
labels_k = labels[keep_idx]
raw_k = [raw[i] for i in keep_idx]
print(f'rows after join: {len(raw_k):,}')

min_dist = np.array([r['min_dist'] for r in raw_k])
was_clicked = np.array([r['was_clicked'] for r in raw_k], dtype=bool)
approached = min_dist < 100
subset = approached & ~was_clicked
print(f'approached ∧ ¬clicked: {int(subset.sum()):,}')
print(f'  deferred (NB22):    {int((subset & labels_k).sum()):,}')
print(f'  eval-rejected:      {int((subset & ~labels_k).sum()):,}')

y = labels_k[subset].astype(int)
participants = np.array([r['trial_id'].split('-')[0] for r in raw_k])[subset]

def feat_matrix(names):
    return np.array([[float(f.get(n, 0.0) or 0.0) for n in names] for f in feat_rows])[subset]

X_A   = feat_matrix(FEATURES_A)
X_B   = feat_matrix(FEATURES_B)
X_C   = feat_matrix(FEATURES_C)
X_AC  = np.hstack([X_A, X_C])
X_BC  = np.hstack([X_B, X_C])
X_ABC = np.hstack([X_A, X_B, X_C])
print(f'matrix shapes: A {X_A.shape}, B {X_B.shape}, C {X_C.shape}, '
      f'A∪C {X_AC.shape}, B∪C {X_BC.shape}, A∪B∪C {X_ABC.shape}')

rows after join: 13,390
approached ∧ ¬clicked: 2,351
  deferred (NB22):    1,916
  eval-rejected:      435
matrix shapes: A (2351, 4), B (2351, 4), C (2351, 7), A∪C (2351, 11), B∪C (2351, 11), A∪B∪C (2351, 15)


In [4]:
# ── LOPO helper: returns pooled AUC + per-participant AUC vector ────
def lopo_auc(X, y, groups):
    pipe = lambda: Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(max_iter=5000, class_weight='balanced', C=1.0)),
    ])
    gkf = GroupKFold(n_splits=len(np.unique(groups)))
    pred = np.zeros(len(y), dtype=float)
    per_p_auc = []
    for train_idx, test_idx in gkf.split(X, y, groups=groups):
        m = pipe()
        m.fit(X[train_idx], y[train_idx])
        p = m.predict_proba(X[test_idx])[:, 1]
        pred[test_idx] = p
        if len(np.unique(y[test_idx])) == 2:
            per_p_auc.append(roc_auc_score(y[test_idx], p))
    pooled = roc_auc_score(y, pred)
    return pooled, np.array(per_p_auc)

print('helper defined')

helper defined


In [5]:
# ── Compute AUC for each feature set ─────────────────────────────────
results = {}
for tag, X in [('A bands (NB28)', X_A),
               ('B continuous viewport', X_B),
               ('C trajectory', X_C),
               ('A ∪ C', X_AC),
               ('B ∪ C', X_BC),
               ('A ∪ B ∪ C', X_ABC)]:
    pooled, per_p = lopo_auc(X, y, participants)
    results[tag] = (pooled, per_p)
    print(f'  {tag:28s}  pooled AUC = {pooled:.4f}   '
          f'per-p mean = {per_p.mean():.4f} ± {per_p.std(ddof=1):.4f} (n={len(per_p)})')

  A bands (NB28)                pooled AUC = 0.7990   per-p mean = 0.8051 ± 0.1272 (n=47)


  B continuous viewport         pooled AUC = 0.7978   per-p mean = 0.8107 ± 0.1204 (n=47)


  C trajectory                  pooled AUC = 0.7496   per-p mean = 0.7559 ± 0.1252 (n=47)


  A ∪ C                         pooled AUC = 0.8104   per-p mean = 0.8179 ± 0.1138 (n=47)


  B ∪ C                         pooled AUC = 0.8168   per-p mean = 0.8293 ± 0.1122 (n=47)


  A ∪ B ∪ C                     pooled AUC = 0.8200   per-p mean = 0.8326 ± 0.1090 (n=47)


In [6]:
# ── Paired per-participant Wilcoxon ──────────────────────────────────
def paired(tag_a, tag_b):
    _, a = results[tag_a]
    _, b = results[tag_b]
    m = min(len(a), len(b))
    a, b = a[:m], b[:m]
    d = a - b
    w = wilcoxon(a, b, alternative='greater')
    print(f'  {tag_a:28s} > {tag_b:22s}  Δ = {d.mean():+.4f} ± {d.std(ddof=1):.4f}   '
          f'{int((a >= b).sum())}/{len(a)}   p = {w.pvalue:.4f}')
    return {'delta': float(d.mean()), 'sd': float(d.std(ddof=1)),
            'a_ge_b': int((a >= b).sum()), 'n': len(a),
            'W': float(w.statistic), 'p': float(w.pvalue)}

K_paired = {}
K_paired['C_beats_A'] = paired('C trajectory', 'A bands (NB28)')
K_paired['C_beats_B'] = paired('C trajectory', 'B continuous viewport')
K_paired['AC_beats_A'] = paired('A ∪ C', 'A bands (NB28)')
K_paired['BC_beats_B'] = paired('B ∪ C', 'B continuous viewport')
K_paired['ABC_beats_BC'] = paired('A ∪ B ∪ C', 'B ∪ C')

  C trajectory                 > A bands (NB28)          Δ = -0.0492 ± 0.1024   14/47   p = 0.9993
  C trajectory                 > B continuous viewport   Δ = -0.0548 ± 0.1127   12/47   p = 0.9997
  A ∪ C                        > A bands (NB28)          Δ = +0.0128 ± 0.0417   29/47   p = 0.0368
  B ∪ C                        > B continuous viewport   Δ = +0.0185 ± 0.0456   36/47   p = 0.0031
  A ∪ B ∪ C                    > B ∪ C                   Δ = +0.0033 ± 0.0232   29/47   p = 0.2178


In [7]:
# ── K-value synthesis ────────────────────────────────────────────────
K1 = int(subset.sum())
K2 = int((subset & labels_k).sum())
K3 = int((subset & ~labels_k).sum())
K4_A  = results['A bands (NB28)'][0]
K5_B  = results['B continuous viewport'][0]
K6_C  = results['C trajectory'][0]
K7_AC = results['A ∪ C'][0]
K8_BC = results['B ∪ C'][0]
K9_ABC = results['A ∪ B ∪ C'][0]

print('── Key Claims (transcribe into update_key_claims.py NB30_BODY) ──')
print(f'K1  rows (approached ∧ ¬clicked)           : {K1:,}')
print(f'K2  deferred (NB22)                        : {K2:,}')
print(f'K3  eval-rejected                          : {K3:,}')
print(f'K4  Pooled AUC, A bands                    : {K4_A:.4f}')
print(f'K5  Pooled AUC, B continuous viewport      : {K5_B:.4f}')
print(f'K6  Pooled AUC, C trajectory               : {K6_C:.4f}')
print(f'K7  Pooled AUC, A ∪ C                      : {K7_AC:.4f}')
print(f'K8  Pooled AUC, B ∪ C                      : {K8_BC:.4f}')
print(f'K9  Pooled AUC, A ∪ B ∪ C                  : {K9_ABC:.4f}')
print()
print(f'K10 C − A  (trajectory alone vs bands)     : Δ = {K_paired["C_beats_A"]["delta"]:+.4f}, '
      f'p = {K_paired["C_beats_A"]["p"]:.4f} ({K_paired["C_beats_A"]["a_ge_b"]}/{K_paired["C_beats_A"]["n"]})')
print(f'K11 C − B  (trajectory alone vs viewport)  : Δ = {K_paired["C_beats_B"]["delta"]:+.4f}, '
      f'p = {K_paired["C_beats_B"]["p"]:.4f} ({K_paired["C_beats_B"]["a_ge_b"]}/{K_paired["C_beats_B"]["n"]})')
print(f'K12 A∪C − A (trajectory added to bands)    : Δ = {K_paired["AC_beats_A"]["delta"]:+.4f}, '
      f'p = {K_paired["AC_beats_A"]["p"]:.4f} ({K_paired["AC_beats_A"]["a_ge_b"]}/{K_paired["AC_beats_A"]["n"]})')
print(f'K13 B∪C − B (trajectory added to viewport) : Δ = {K_paired["BC_beats_B"]["delta"]:+.4f}, '
      f'p = {K_paired["BC_beats_B"]["p"]:.4f} ({K_paired["BC_beats_B"]["a_ge_b"]}/{K_paired["BC_beats_B"]["n"]})    ← headline')
print(f'K14 A∪B∪C − B∪C (bands add to B+C)         : Δ = {K_paired["ABC_beats_BC"]["delta"]:+.4f}, '
      f'p = {K_paired["ABC_beats_BC"]["p"]:.4f} ({K_paired["ABC_beats_BC"]["a_ge_b"]}/{K_paired["ABC_beats_BC"]["n"]})')

── Key Claims (transcribe into update_key_claims.py NB30_BODY) ──
K1  rows (approached ∧ ¬clicked)           : 2,351
K2  deferred (NB22)                        : 1,916
K3  eval-rejected                          : 435
K4  Pooled AUC, A bands                    : 0.7990
K5  Pooled AUC, B continuous viewport      : 0.7978
K6  Pooled AUC, C trajectory               : 0.7496
K7  Pooled AUC, A ∪ C                      : 0.8104
K8  Pooled AUC, B ∪ C                      : 0.8168
K9  Pooled AUC, A ∪ B ∪ C                  : 0.8200

K10 C − A  (trajectory alone vs bands)     : Δ = -0.0492, p = 0.9993 (14/47)
K11 C − B  (trajectory alone vs viewport)  : Δ = -0.0548, p = 0.9997 (12/47)
K12 A∪C − A (trajectory added to bands)    : Δ = +0.0128, p = 0.0368 (29/47)
K13 B∪C − B (trajectory added to viewport) : Δ = +0.0185, p = 0.0031 (36/47)    ← headline
K14 A∪B∪C − B∪C (bands add to B+C)         : Δ = +0.0033, p = 0.2178 (29/47)


In [8]:
# ── Leakage check: truncate at click time, recompute C features ──
# Concern: exit_velocity captures the scroll velocity when the AOI last left
# the viewport. If post-click settle-scroll happens while a not-clicked AOI
# is still visible, that settle could inflate exit_velocity. The scored
# population is approached ∧ ¬clicked — so the SCORED AOI is never the click
# target, but a settle-scroll on the *click target* can still contribute.
# Verify by recomputing with t_end = click_t.

from data_loader import load_mouse_events as _load_me

click_t_by_trial = {}
for tid in trials:
    _, _, clicks = _load_me(tid)
    if clicks:
        click_t_by_trial[tid] = min(c[0] for c in clicks)

print(f"trials with a click timestamp: {len(click_t_by_trial):,}")
print("recomputing features with t_end = click_t (leakage-truncated)...")
per_trial_trunc = {}
for i, tid in enumerate(trials):
    if tid not in click_t_by_trial:
        continue
    feats = compute_features_for_trial(tid, n_positions=10, max_t=click_t_by_trial[tid])
    if feats is not None:
        per_trial_trunc[tid] = feats
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(trials)}")
print(f"truncated feature set computed for {len(per_trial_trunc):,} trials")

# Rebuild feature matrices with truncated per_trial
feat_rows_t = []
keep_idx_t = []
for i, r in enumerate(raw):
    tid = r['trial_id']; pos = r['position']
    if tid not in per_trial_trunc or pos >= 10:
        continue
    feat_rows_t.append(per_trial_trunc[tid][pos])
    keep_idx_t.append(i)
keep_idx_t = np.array(keep_idx_t)
labels_t = labels[keep_idx_t]
raw_t = [raw[i] for i in keep_idx_t]
md_t = np.array([r['min_dist'] for r in raw_t])
wc_t = np.array([r['was_clicked'] for r in raw_t], dtype=bool)
subset_t = (md_t < 100) & ~wc_t
y_t = labels_t[subset_t].astype(int)
parts_t = np.array([r['trial_id'].split('-')[0] for r in raw_t])[subset_t]

def feat_matrix_t(names):
    return np.array([[float(f.get(n, 0.0) or 0.0) for n in names] for f in feat_rows_t])[subset_t]

X_B_t  = feat_matrix_t(FEATURES_B)
X_C_t  = feat_matrix_t(FEATURES_C)
X_BC_t = np.hstack([X_B_t, X_C_t])
print(f"\nleakage-truncated subset size: {int(subset_t.sum()):,} "
      f"(deferred {int((subset_t & labels_t).sum())}, eval-rej {int((subset_t & ~labels_t).sum())})")

pooled_B_t,  per_p_B_t  = lopo_auc(X_B_t,  y_t, parts_t)
pooled_BC_t, per_p_BC_t = lopo_auc(X_BC_t, y_t, parts_t)
print(f"  B  (truncated) pooled AUC = {pooled_B_t:.4f}   per-p = {per_p_B_t.mean():.4f}")
print(f"  B∪C (truncated) pooled AUC = {pooled_BC_t:.4f}   per-p = {per_p_BC_t.mean():.4f}")

from scipy.stats import wilcoxon as _wx
delta_t = per_p_BC_t - per_p_B_t
w_t = _wx(per_p_BC_t, per_p_B_t, alternative='greater')
print(f"\nK15 — leakage-truncated B∪C > B:")
print(f"  Δ = {delta_t.mean():+.4f} ± {delta_t.std(ddof=1):.4f}   "
      f"{int((per_p_BC_t >= per_p_B_t).sum())}/{len(per_p_BC_t)}   p = {w_t.pvalue:.4f}")
print(f"  vs. K13 full-trial Δ = {K_paired['BC_beats_B']['delta']:+.4f}  (drift |Δ−Δ_trunc| = {abs(K_paired['BC_beats_B']['delta'] - delta_t.mean()):.4f})")
if abs(K_paired['BC_beats_B']['delta'] - delta_t.mean()) < 0.005:
    print("  VERDICT: truncated Δ is within 0.005 of full-trial Δ — leakage not detected.")
else:
    print("  VERDICT: truncated Δ deviates by >= 0.005 from full-trial Δ — revisit the leakage concern.")


trials with a click timestamp: 2,338
recomputing features with t_end = click_t (leakage-truncated)...


  500/2339


  1000/2339


  1500/2339


  2000/2339


truncated feature set computed for 2,338 trials

leakage-truncated subset size: 2,350 (deferred 1916, eval-rej 434)


  B  (truncated) pooled AUC = 0.7923   per-p = 0.8073
  B∪C (truncated) pooled AUC = 0.8130   per-p = 0.8273

K15 — leakage-truncated B∪C > B:
  Δ = +0.0200 ± 0.0493   34/47   p = 0.0038
  vs. K13 full-trial Δ = +0.0185  (drift |Δ−Δ_trunc| = 0.0014)
  VERDICT: truncated Δ is within 0.005 of full-trial Δ — leakage not detected.


## Interpretation guide

- **K11 (C > B) significant** would say trajectory alone beats continuous
  viewport analytics. Surprising if true; more likely directionally
  negative because trajectory in isolation throws away visibility duration.
- **K13 (B∪C > B) is the headline.** Positive and significant means scroll
  trajectory features add measurable signal on top of the continuous-viewport baseline (B) — a second cursor-free evaluation
  probe. If null, NB17's scroll null tightens to: even under incremental-AUC testing
  against viewport analytics, trajectory adds nothing for desktop SERPs.
- **K12 (A∪C > A)** is the same question against the NB28 banded baseline.
  Both K12 and K13 positive means trajectory adds regardless of which
  viewport operationalization you pair it with.
- **K14 (A∪B∪C > B∪C)** bounds whether bands add on top of continuous
  viewport + trajectory. If null, bands are redundant with continuous
  viewport once trajectory is in the feature set — i.e. the banded
  discretization is a research artifact, not a load-bearing signal.

## What to report (filled in after execution)

- K4–K9 pooled AUCs per feature set.
- K13 headline: B∪C − B paired Δ, one-sided Wilcoxon p, win/n.
- K14 follow-up: does bands add anything on top?

**Caveat.** This is 47 LAB participants on AdSERP desktop. Generalization
to mobile and to feed-style layouts is an open question — the features
themselves transfer (scroll + DOM bboxes are available in both regimes)
but the effect sizes will differ. The cursor-approach feature set is
still AUC 0.792 alone on this split and likely dominates on desktop;
the value of scroll trajectory is specifically on channels where cursor
is unavailable.
